## Prerequisites

This notebook demonstrates how to invoke a **deployed real-time inference endpoint** on Azure for the Clinical De-identification for French model. Follow the steps below in order before running this notebook.

### Deployment Sequence

1. **Acquire the Azure Application Offer** — Subscribe to the *Clinical De-identification for French* offer in the [Azure Marketplace](https://marketplace.azure.com) to obtain the Solution Template.
2. **Launch guided deployment** — From the Marketplace listing, start the guided deployment flow for the solution.
3. **Supply configuration** — Complete the guided UX forms (defined by the offer's `createUiDefinition`) with the required parameters.
4. **Deploy resources** — Confirm deployment; Azure provisions the resources defined in the offer's `mainTemplate.json`.
5. **Wait for deployment to succeed** — Verify the deployment reaches a *Succeeded* state before proceeding. Note the endpoint name from the deployment output.
6. **Run this notebook** — Execute the cells below against the real-time inference endpoint provisioned by the deployment.

### Environment

- **Notebook runtime**: This notebook is intended to run in **Azure Machine Learning workspace notebooks** (e.g. via an Azure ML Studio compute instance or Studio notebooks).
- **Jupyter kernel**: Use a kernel with the `azure.ai.ml` Python package installed, for example **Python 3.10 — SDK v2** (`python310-sdkv2`).
- **Authentication**: The kernel must be authenticated to Azure so that Azure ML SDK calls succeed (e.g. `DefaultAzureCredential()`). The authenticated identity must be granted Azure RBAC permissions appropriate for reading the workspace and online endpoint configuration, and for retrieving endpoint credentials. Consult your organization's access-policy guidance and the [Azure ML endpoint credential documentation](https://learn.microsoft.com/azure/machine-learning/how-to-manage-endpoints) for details.
- **Network access**: The notebook's compute instance must be able to reach the endpoint's `scoring_uri` (no outbound firewall or VNET restrictions that block it).


## Clinical Deidentification French

Deidentification is essential for safeguarding patient privacy in clinical data, including texts, PDFs, images, and DICOM files containing Protected Health Information (PHI). PHI encompasses various health-related data, including common identifiers such as name, address, birth date, and Social Security Number.

- **Model**: [fr.deid_obfuscated](https://nlp.johnsnowlabs.com/2023/06/17/clinical_deidentification_fr.html)
- **Model Description**: This pipeline can be used to deidentify PHI information from medical texts in French. The pipeline can mask and obfuscate the following entities: DATE, AGE, SEX, PROFESSION, ORGANIZATION, PHONE, E-MAIL, ZIP, STREET, CITY, COUNTRY, PATIENT, DOCTOR, HOSPITAL, MEDICALRECORD, SSN, IDNUM, ACCOUNT, PLATE, USERNAME, URL, and IPADDR.


In [ ]:
ENDPOINT_NAME = "<CUSTOMER_TO_PUT_DEPLOYED_ENDPOINT_NAME>"

### Input Format

To use the model, you need to provide input in one of the following supported formats:

#### JSON Format

Provide input as JSON. We support two variations within this format:

1. **Array of Text Documents**: 
   Use an array containing multiple text documents. Each element represents a separate text document.

   ```json
   {
       "text": [
           "Text document 1",
           "Text document 2",
           ...
       ]
   }

    ```

2. **Single Text Document**:
   Provide a single text document as a string.


   ```json
    {
        "text": "Single text document"
    }
   ```

#### JSON Lines (JSONL) Format

Provide input in JSON Lines format, where each line is a JSON object representing a text document.

```
{"text": "Text document 1"}
{"text": "Text document 2"}
```

### Important Parameter

- **masking_policy**: `str`

    Users can select a masking policy to determine how sensitive entities are handled:

    Example: "**Nombre: Danilo. Apellidos: Ramos Portillo. NHC: 4376372. NASS: 35 61051794 56. Domicilio: Calle Fernando Higueras, 27, 7B.**"

    - **masked**: Default policy that masks entities with their type.

      -> 'Nombre: `<PATIENT>`. Apellidos: `<PATIENT>`. NHC: `<SSN>`. NASS: `<ID>`. Domicilio: `<PATIENT>`, `<AGE>`, 7B.'

    - **obfuscated**: Replaces sensitive entities with random values of the same type.

      -> 'Nombre: `Aurora Garrido Paez`.Apellidos: `Antonio González Cuevas`.NHC: `BBBBBBBBQR648597`.NASS: `48.124.111S`. Domicilio: `Aurora Garrido Paez`, `36`, 7B.'

    - **masked_fixed_length_chars**: Masks entities with a fixed length of asterisks (\*).

      -> 'Nombre: `****`. Apellidos: `****`. NHC: `****`. NASS: `****`. Domicilio: `****`, `****`, 7B.'

    - **masked_with_chars**: Masks entities with asterisks (\*) matching the length of each entity.

      -> 'Nombre: [`******`]. Apellidos: [`*************`]. NHC: [`*******`]. NASS: [`*************`]. Domicilio: [`**********************`], `**`, 7B.'
    
You can specify these parameters in the input as follows:

```json
{
    "text": [
        "Text document 1",
        "Text document 2"
    ],
    "masking_policy": "masked"
}
```

> **Note:** Demonstration inputs used with this endpoint should consist of approved synthetic or properly de-identified data. Never submit real protected health information (PHI) without explicit authorization and the required safeguarding controls.


### Authentication and Endpoint Lookup

The cell below uses `DefaultAzureCredential()` to authenticate to Azure, then retrieves the endpoint's scoring URI and primary access key from the workspace configuration. If you see authentication errors, verify that you are logged into Azure Cloud Shell / your compute instance and that your identity has sufficient role assignments on the workspace.


In [2]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
import requests
import json


ml_client = MLClient.from_config(DefaultAzureCredential())
endpoint_url = ml_client.online_endpoints.get(ENDPOINT_NAME).scoring_uri
key = ml_client.online_endpoints.get_keys(ENDPOINT_NAME).primary_key


def invoke_realtime_endpoint(record, content_type="application/json", accept="application/json"):
    response = requests.post(
        endpoint_url,
        headers={
            "Authorization": f"Bearer {key}",
            "Content-Type": content_type,
        },
        data=json.dumps(record) if content_type == "application/json" else record,
     
    )

    response_body = response.content.decode("utf-8")

    if accept == "application/json":
        return json.loads(response_body)
    elif accept == "application/jsonlines":
        return response_body
    else:
        raise ValueError(f"Unsupported accept type: {accept}")

Found the config file in: ./config.json


### Sample Data (Synthetic — for demonstration only)

The following cells define sample clinical text documents used in the examples below. When using this notebook with your own data, ensure all inputs are approved synthetic or properly de-identified. Never submit real protected health information (PHI) without explicit authorization and appropriate controls.


In [12]:
docs = [
'''PRENOM : Jean, NOM : Dubois, NUMÉRO DE SÉCURITÉ SOCIALE : 1780160471058, ADRESSE : 18 Avenue Matabia... 
PRENOM : Jean, NOM : Dubois, NUMÉRO DE SÉCURITÉ SOCIALE : 1780160471058, ADRESSE : 18 Avenue Matabiau, VILLE : Grenoble
CODE POSTAL : 38000, DATE DE NAISSANCE : 03/03/1946, Âge : 70 ans , Sexe : H, NUMÉRO DE TÉLÉPHONE : 06-777413307, COURRIEL : jdubois@hotmail.fr, DATE D'ADMISSION : 12/12/2016
MÉDÉCIN : Dr Michel Renaud, RAPPORT CLINIQUE : 70 ans, retraité, sans allergie médicamenteuse connue, qui présente comme antécédents : ancien accident du travail avec fractures vertébrales et des côtes ; opéré de la maladie de Dupuytren à la main droite et d'un pontage ilio-fémoral gauche ; diabète de type II, hypercholestérolémie et hyperuricémie ; alcoolisme actif, fume 20 cigarettes / jour.
Jean Dubois nous a été adressé car il présentait une hématurie macroscopique postmictionnelle à une occasion et une microhématurie persistante par la suite, avec une miction normale.
L'examen physique a montré un bon état général, avec un abdomen et des organes génitaux normaux ; le toucher rectal était compatible avec un adénome de la prostate de grade I/IV.
L'analyse d'urine a montré 4 globules rouges/champ et 0-5 leucocytes/champ ; le reste du sédiment était normal.
Hémogramme normal ; la biochimie a montré une glycémie de 169 mg/dl et des triglycérides de 456 mg/dl ; les fonctions hépatiques et rénales étaient normales. PSA de 1,16 ng/ml.
ADDRESSÉ À : Dre Marie Breton - Centre Hospitalier de Bellevue Service D'Endocrinologie et de Nutrition - Rue Paulin Bussières, 38000 Grenoble
COURRIEL : mariebreton@chb.fr
NUMÉRO DE TÉLÉPHONE : 06-34567890''',

'''PRENOM : Éric, NOM : Lejeune, NUMÉRO DE SÉCURITÉ SOCIALE : 2730235238020, ADRESSE : 74 Boulevard Rio... 
PRENOM : Éric, NOM : Lejeune, NUMÉRO DE SÉCURITÉ SOCIALE : 2730235238020, ADRESSE : 74 Boulevard Riou, VILLE : Sainte Annenec
CODE POSTAL : 49722, DATE DE NAISSANCE : 10/06/1980, Âge : 38 ans , Sexe : H, NUMÉRO DE TÉLÉPHONE : 06-90123456. COURRIEL : elej@free.fr, DATE D'ADMISSION : 08/11/2018
MÉDÉCIN : Dre Martine Da Costa, RAPPORT CLINIQUE : 
Antécédents : Éric Lejeune de 38 ans, évalué en consultation pour une coxalgie droite d'un an et demi d'évolution, de caractéristiques mécaniques. Éric Lejeune a signalé une douleur à l'exercice et une légère limitation de la mobilité dans certaines postures. Mécanicien de profession, pendant son temps libre, Éric Lejeune faisait du vélo (40 km par jour le week-end), courait 3 fois par semaine pendant 50 minutes et marchait pendant une heure les jours où il ne courait pas. La douleur a progressivement augmenté et elle ne fait actuellement que marcher.
EXAMEN PHYSIQUE : L'examen physique a révélé une douleur à la palpation dans le tiers moyen de la région inguinale droite avec une douleur à la flexion, à l'adduction et à la rotation, surtout à la rotation interne (manœuvre de choc positif). L'équilibre de l'articulation de la hanche est de 110o de flexion bilatérale, 10o de rotation interne droite, 40o de rotation interne gauche et 30o de rotation externe droite, 40o de rotation externe gauche. L'équilibre musculaire sur l'échelle de Daniels est 5/5 global bilatéral. Pas de douleur à la palpation des points sciatiques et les manœuvres de Valleix, Fabere, Lasegue et Bragard sont négatives. Pas non plus de douleur à la palpation de la bandelette iliotibiale et le test d'Ober était négatif. A l'examen de la marche, nous avons observé une absence de claudication et la posture monopodale alternée droite-gauche était stable, avec un Trendelenburg négatif. Le patient avait des douleurs lors de la marche avec une flexion forcée de la hanche et du genou.
Examens complémentaires : Une radiologie ordinaire antéropostérieure (AP) a montré une déformation en forme de bosse dans la région fémorale cervicale droite. L'hémogramme, la biochimie, les hormones thyroïdiennes, les anticorps et les réactifs de phase aiguë requis étaient normaux.
Au vu de ces résultats, une IRM de la hanche a été demandée, qui a montré de légères modifications sous-chondrales, avec un œdème osseux dans la hanche droite, avec une irrégularité corticale et une légère synovite dans l'interligne articulaire, avec une distension de la bourse ilio-psoas. La tête fémorale présente un petit îlot osseux et l'angle alpha est supérieur à 50o. L'arthrographie montre un pincement de l'interligne articulaire antéro-supérieur et postéro-inférieur, un col fémoral ostéophytique naissant avec de petites lésions sous-chondrales antéro-supérieures de la tête fémorale et de petits foyers d'œdème dans l'acétabulum antéro-supérieur. Pas de nécrose avasculaire ni de lésions du labrum. Une lésion kystique juxta-articulaire est également observée en regard du grand trochanter.
Evolution : Avec le diagnostic de syndrome de conflit fémoroacétabulaire de type LEVA (également connu sous le nom de CAM), le patient s'est vu prescrire un traitement par magnétothérapie (15 séances) pour améliorer les symptômes de la douleur et le trophisme osseux.
Éric Lejeune a été adressé à l'unité d'arthroscopie du service de traumatologie pour une évaluation de la chirurgie arthroscopique.
Diagnostic principal : syndrome du conflit fémoroacétabulaire de type LEVA.
TRAITEMENT : Un traitement pharmacologique est prescrit avec une combinaison de glucosamine-chondroïtine sulfate par voie orale 400 mg toutes les 12 heures pendant 3 mois et des anti-inflammatoires non stéroïdiens si nécessaire en cas de douleur. Elle a pour consigne de se reposer dans les sports pour éviter tout choc à l'articulation, avec une activité active contrôlée, en recommandant de ne pas dépasser 90 degrés de flexion de la hanche et d'éviter la rotation. Prenez rendez-vous pour un suivi une fois les sulfates de glucosamine-chondroïtine terminés afin de constater une amélioration clinique. Appelez le numéro des soins et des rendez-vous précédents de l'hôpital : 0470892301.
ADDRESSÉ À : Pr Diane Martinez - Centre Hospitalier Saint Louis - 30 Rue Garcia, 49000 Angers
COURRIEL : dmartinez@chsl.fr
NUMÉRO DE TÉLÉPHONE : 06-44006512'''

]


sample_text = """PRENOM : Aurélie
NOM : Perrier
NUMÉRO DE SÉCURITÉ SOCIALE : 2190150502008
ADRESSE : 8 Rue Faivre
VIL... 
PRENOM : Aurélie
NOM : Perrier
NUMÉRO DE SÉCURITÉ SOCIALE : 2190150502008
ADRESSE : 8 Rue Faivre
VILLE : Bousquet
CODE POSTAL : 11631
DATE DE NAISSANCE : 22/10/1960
Âge : 44 ans 
Sexe : Femme
COURRIEL : aurelieprr@orange.fr
DATE D'ADMISSION : 31/12/2004
MÉDÉCIN : Dr Alain Bouvet
RAPPORT CLINIQUE : 
Motif de l'admission : Aurélie Perrier âgée de 44 ans, adressée à notre unité en raison d'une augmentation de volume dans la région zygomatique et d'une limitation progressive de l'ouverture de la bouche depuis 10 mois. Les antécédents médicaux n'étaient pas pertinents. 
Examen : L'examen physique extraoral a montré une asymétrie faciale due à une augmentation de volume de la joue gauche, de limites diffuses, de consistance osseuse, indolore, avec une peau d'apparence normale. L'ouverture de la bouche était de 30 mm. L'articulation temporomandibulaire (ATM) était normale à la palpation, avec des bruits ou des douleurs à l'ouverture. Sur l'orthopantomographie, nous avons observé une hypertrophie du processus coronoïde gauche. L'hypothèse diagnostique était une tumeur coronoïde.
Tests complémentaires et évolution : Une incision a été pratiquée sur le bord antérieur du ramus mandibulaire, en désinsérant toutes les insertions du muscle temporal. Une pseudocapsule fibreuse a été trouvée autour de l'élargissement, qui a été libérée. Une coronoïdectomie a été effectuée. L'ouverture buccale a été immédiatement restaurée à 43 mm. L'évolution postopératoire s'est déroulée sans incident avec une sortie de l'hôpital après 48 heures.
L'étude histopathologique a signalé la présence de fibres, d'os de néoformation et de tissu cartilagineux hyalin. Les contrôles ultérieurs se sont déroulés sans incident. La sortie définitive a été donnée à 10 mois, avec un remodelage presque complet de la déformation des joues.
ADDRESSÉ À : Dr Joseph Pasquier - Centre Hospitalier de Carcassonne - 1060 Chemin de la Madeleine, 11000 Carcassonne
NUMÉRO DE TÉLÉPHONE : 06-71105388
COURRIEL : pasquier@orange.fr"""

### JSON

#### Example 1: masked (default-policy)

In [22]:
input_json_data = {"text": sample_text}
response_json = invoke_realtime_endpoint(input_json_data, content_type="application/json", accept="application/json")
print(response_json["predictions"])

PRENOM : <PATIENT>
NOM : <PATIENT>
NUMÉRO DE SÉCURITÉ SOCIALE : <SSN>
ADRESSE : <CITY>... 
PRENOM : <PATIENT>
NOM : <PATIENT>
NUMÉRO DE SÉCURITÉ SOCIALE : <SSN>
ADRESSE : <STREET>
VILLE : <CITY>
CODE POSTAL : <ZIP>
DATE DE NAISSANCE : <DATE>
Âge : <AGE> 
Sexe : <SEX>
COURRIEL : <E-MAIL>
DATE D'ADMISSION : <DATE>
MÉDÉCIN : <DOCTOR>
RAPPORT CLINIQUE : 
Motif de l'admission : <PATIENT> âgée de <AGE>ans, adressée à notre unité en raison d'une augmentation de volume dans la région zygomatique et d'une limitation progressive de l'ouverture de la bouche depuis 10 mois. <CITY> antécédents médicaux n'étaient pas pertinents. 
Examen : L'examen physique extraoral a montré une asymétrie faciale due à une augmentation de volume de la joue gauche, de limites diffuses, de consistance osseuse, indolore, avec une peau d'apparence normale. L'ouverture de la bouche était de 30 mm. L'articulation temporomandibulaire (ATM) était normale à la palpation, avec des bruits ou des douleurs à l'ouverture. Sur l'o

#### Example 2: obfuscated

In [21]:
input_json_data = {"text": sample_text, "masking_policy": "obfuscated"}
response_json = invoke_realtime_endpoint(input_json_data, content_type="application/json", accept="application/json")
print(response_json["predictions"])

PRENOM : Mme Lacombe
NOM : M. Besson
NUMÉRO DE SÉCURITÉ SOCIALE : 9021061619113
ADRESSE : Faivre-sur-andre... 
PRENOM : Mme Lacombe
NOM : M. Besson
NUMÉRO DE SÉCURITÉ SOCIALE : 9021061619113
ADRESSE : 77, Rue Timothée Bourgeois
VILLE : Michaud-sur-carpentier
CODE POSTAL : 00580
DATE DE NAISSANCE : 22/10/1960
Âge : 48 ans 
Sexe : Homme
COURRIEL : margotpichon@ifrance.com
DATE D'ADMISSION : 31/12/2004
MÉDÉCIN : Dr. Nicolas
RAPPORT CLINIQUE : 
Motif de l'admission : Mme Thibaut Du Hebert âgée de 48ans, adressée à notre unité en raison d'une augmentation de volume dans la région zygomatique et d'une limitation progressive de l'ouverture de la bouche depuis 10 mois. Saint Nicolas antécédents médicaux n'étaient pas pertinents. 
Examen : L'examen physique extraoral a montré une asymétrie faciale due à une augmentation de volume de la joue gauche, de limites diffuses, de consistance osseuse, indolore, avec une peau d'apparence normale. L'ouverture de la bouche était de 30 mm. L'articulation te

### JSON Lines

In [ ]:
def create_jsonl(records, masking_policy=None):
    json_records = []

    if isinstance(records, str):
        records = [records]

    for text in records:
        record = {"text": text}

        if masking_policy is not None:
            record["masking_policy"] = masking_policy
        json_records.append(record)

    json_lines = '\n'.join(json.dumps(record, ensure_ascii=False) for record in json_records)
    return json_lines


#### Example 1: masked (default-policy)

In [25]:
input_jsonl_data = create_jsonl(docs, masking_policy="masked")
data = invoke_realtime_endpoint(input_jsonl_data, content_type="application/jsonlines" , accept="application/jsonlines" )
print(data)

{"predictions": "PRENOM : Jean, NOM : Dubois, NUM\u00c9RO DE S\u00c9CURIT\u00c9 SOCIALE : <SSN>, ADRESSE : 18 Avenue Matabia... \nPRENOM : Jean, NOM : Dubois, NUM\u00c9RO DE S\u00c9CURIT\u00c9 SOCIALE : <SSN>, ADRESSE : <STREET>, VILLE : <CITY>\nCODE POSTAL : <ZIP> DATE DE NAISSANCE : <DATE>, \u00c2ge : <AGE> , Sexe : <SEX>, NUM\u00c9RO DE T\u00c9L\u00c9PHONE : <PHONE>, COURRIEL : <E-MAIL>, DATE D'ADMISSION : <DATE>\nM\u00c9D\u00c9CIN : <DOCTOR>, RAPPORT CLINIQUE : <AGE>ans, retrait\u00e9, sans allergie m\u00e9dicamenteuse connue, qui pr\u00e9sente comme ant\u00e9c\u00e9dents : ancien accident du travail avec fractures vert\u00e9brales et des c\u00f4tes ; op\u00e9r\u00e9 de la maladie de Dupuytren \u00e0 la main droite et d'un pontage ilio-f\u00e9moral gauche ; diab\u00e8te de type II, hypercholest\u00e9rol\u00e9mie et hyperuric\u00e9mie ; alcoolisme actif, fume 20 cigarettes / jour.\n<PATIENT> nous a \u00e9t\u00e9 adress\u00e9 car <SEX> pr\u00e9sentait une h\u00e9maturie macroscopique

#### Example 2: obfuscated

In [26]:
input_jsonl_data = create_jsonl(docs, masking_policy="obfuscated")
data = invoke_realtime_endpoint(input_jsonl_data, content_type="application/jsonlines" , accept="application/jsonlines" )
print(data)

{"predictions": "PRENOM : Jean, NOM : Dubois, NUM\u00c9RO DE S\u00c9CURIT\u00c9 SOCIALE : 0431051740163, ADRESSE : 18 Avenue Matabia... \nPRENOM : Jean, NOM : Dubois, NUM\u00c9RO DE S\u00c9CURIT\u00c9 SOCIALE : 0431051740163, ADRESSE : Boulevard De Vallet, VILLE : Sainte Martin\nCODE POSTAL : 83111 DATE DE NAISSANCE : 03/03/1946, \u00c2ge : 75 ans , Sexe : FEMME, NUM\u00c9RO DE T\u00c9L\u00c9PHONE : 15-444708814, COURRIEL : georgeslemonnier@live.com, DATE D'ADMISSION : 12/12/2016\nM\u00c9D\u00c9CIN : Dr. Sauvage, RAPPORT CLINIQUE : 75ans, retrait\u00e9, sans allergie m\u00e9dicamenteuse connue, qui pr\u00e9sente comme ant\u00e9c\u00e9dents : ancien accident du travail avec fractures vert\u00e9brales et des c\u00f4tes ; op\u00e9r\u00e9 de la maladie de Dupuytren \u00e0 la main droite et d'un pontage ilio-f\u00e9moral gauche ; diab\u00e8te de type II, hypercholest\u00e9rol\u00e9mie et hyperuric\u00e9mie ; alcoolisme actif, fume 20 cigarettes / jour.\nMme Alexandrie Brun nous a \u00e9t\u0